Imports and Baseline data: NumPy, pandas, matplotlib, and scikit-learn

In [114]:
## IMPORTS
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt # Used for our plotting
from sklearn.preprocessing import StandardScaler # preprocessing is for data cleaning and preparing, StandardScalar is to replace my residuals as std deviations
from sklearn.ensemble import IsolationForest # ensemble (for ensemble based models) is used to create small models and combine them, and IsolationForest is to create many random trees that splits the data then combines results

## BASELINE DATA
baseline_dataframe = pd.read_csv("../data/baseline_data.csv")

#Setting cp again and grabbing necessary values for the leak fit
cp = 4180   # specific heat of water, J/(kg*K)
mdot_h_values = baseline_dataframe["mdot_h"].values
Th_i_values = baseline_dataframe["Th_i"].values

baseline_dataframe.head() # Shows the first 5 lines and what we must create residuals for 

,Timestamp,Th_i,Tc_i,mdot_h,mdot_c,Th_o_true,Tc_o_true,Th_o_measured,Tc_o_measured
0,2026-01-01 00:00:00,79.242229,19.903684,1.501541,0.991531,53.042709,59.579353,53.072301,59.506250
1,2026-01-01 00:01:00,79.070233,20.470527,1.523044,1.007011,53.230447,59.551661,52.994693,59.673443
2,2026-01-01 00:02:00,80.248991,20.714503,1.495319,1.015707,53.541251,60.033507,53.689095,60.229359
3,2026-01-01 00:03:00,80.718213,19.625929,1.473627,0.978013,53.600747,60.485320,53.519510,60.647273
4,2026-01-01 00:04:00,79.871143,19.606406,1.492450,1.017070,52.780987,59.358527,52.504071,59.379281


Residuals: The goal here is to create residuals for Th_o and Tc_o for each of the 5 data sets (the four fault scenarios and the baseline), the residuals are made by taking the true physical value and subtracting the measured operating value from the sensors. We want the magnitude of the residual and to analyze a trend over time. During normal operating conditions the value of the residuals should hover around zero and grow with faults.

In [115]:
# Baseline residuals
baseline_dataframe["Th_o_residual"] = baseline_dataframe["Th_o_true"] - baseline_dataframe["Th_o_measured"] #Th_o baseline residual
baseline_dataframe["Tc_o_residual"] = baseline_dataframe["Tc_o_true"] - baseline_dataframe["Tc_o_measured"] #Tc_o baseline residual 

print(baseline_dataframe["Th_o_residual"].describe()) # Using the describe feature in pandas to show necessary statistical values 
## here we see mean is 0.01 and std is 0.2. The mean being 0.01 which is basically zero which is what baseline should be. And the std being 0.2 matches our sensor noise
print()
print(baseline_dataframe["Tc_o_residual"].describe())
## again we see mean is 0.005 and std is 0.21. The mean being 0.005 which is basically zero which is what baseline should be. And the std being 0.2 matches our sensor noise

count    240.000000
mean       0.010167
std        0.200046
min       -0.491439
25%       -0.126550
50%       -0.000941
75%        0.141921
max        0.624040
Name: Th_o_residual, dtype: float64

count    240.000000
mean       0.005109
std        0.210234
min       -0.395571
25%       -0.156739
50%        0.005971
75%        0.150461
max        0.642574
Name: Tc_o_residual, dtype: float64


In [116]:
# Fouling residuals
fouling_dataframe = pd.read_csv("../data/fouling_fault_data.csv")

#Th_o
fouling_dataframe["Th_o_fouling"] = baseline_dataframe["Th_o_true"] - fouling_dataframe["Th_o_measured"]
print(fouling_dataframe["Th_o_fouling"].describe())
# mean is -0.38 meaning a shift has occcured, std is about 4x larger than the baseline, min and max is also a much wider gap
print()
#Tc_o
fouling_dataframe["Tc_o_fouling"] = baseline_dataframe["Tc_o_true"] - fouling_dataframe["Tc_o_measured"]
print(fouling_dataframe["Tc_o_fouling"].describe())
# mean is 0.55 which is a good sign (shift in conditions), and std is 0.9 so about 4.5x larger than baseline

# Also the mean for Th_o and Tc_o has a different sign, this proves the model works as Th_o raises after fouling as there is less heat transfer so if
# the baseline is 53 and it grows to 55 it will be a negative value, vise versa for Tc_o

count    240.000000
mean      -0.387412
std        0.850099
min       -3.229502
25%       -0.895433
50%       -0.324627
75%        0.127544
max        1.890418
Name: Th_o_fouling, dtype: float64

count    240.000000
mean       0.555991
std        0.903966
min       -1.756282
25%       -0.018545
50%        0.542634
75%        1.110862
max        3.105534
Name: Tc_o_fouling, dtype: float64


In [117]:
# Sensor drift residuals
drift_dataframe = pd.read_csv("../data/drift_fault_data.csv")
drift_dataframe.head()

#Th_o
drift_dataframe["Th_o_drift"] = baseline_dataframe["Th_o_true"] - drift_dataframe["Th_o_measured"]
print(drift_dataframe["Th_o_drift"].describe())
# These values makes sense as the drift changes the temp more than fouling so the mean is higher
print()
#Tc_o 
drift_dataframe["Tc_o_drift"] = baseline_dataframe["Tc_o_true"] - drift_dataframe["Tc_o_measured"]
print(drift_dataframe["Tc_o_drift"].describe())
# Tc_o sensor does not drift > behaves as expected

count    240.000000
mean      -1.495667
std        0.903333
min       -3.442607
25%       -2.232821
50%       -1.551291
75%       -0.781449
max        0.367496
Name: Th_o_drift, dtype: float64

count    240.000000
mean      -0.007621
std        0.200691
min       -0.487987
25%       -0.151181
50%       -0.022043
75%        0.113936
max        0.620829
Name: Tc_o_drift, dtype: float64


In [118]:
# Blockage residuals
blockage_dataframe = pd.read_csv("../data/blockage_fault_data.csv")

#Th_o
blockage_dataframe["Th_o_blockage"] = baseline_dataframe["Th_o_true"] - blockage_dataframe["Th_o_measured"]
print(blockage_dataframe["Th_o_blockage"].describe())
print()
#Tc_o
blockage_dataframe["Tc_o_blockage"] = baseline_dataframe["Tc_o_true"] - blockage_dataframe["Tc_o_measured"]
print(blockage_dataframe["Tc_o_blockage"].describe())

#both residuals make sense as there is a sudden jump due to blockage so the std is about 0 until the blockage instantly jumps to a much bigger value

count    240.000000
mean       5.109438
std        6.335879
min       -2.582604
25%       -0.196345
50%        0.699382
75%       12.468006
max       15.057758
Name: Th_o_blockage, dtype: float64

count    240.000000
mean       4.138072
std        5.168897
min       -2.490363
25%       -0.140633
50%        0.711282
75%       10.067458
max       12.816688
Name: Tc_o_blockage, dtype: float64


In [119]:
# Leak residuals
leak_dataframe = pd.read_csv("../data/leak_fault_data.csv")
leak_dataframe.head()

# the energy mismatch I created before is already acting as our residual
print(leak_dataframe["energy_mismatch"].describe())

# Min = 0 shows there is no leak at the start, then is at 12016 W at the end (max) showing gradual linear leakage

count      240.000000
mean      5856.043984
std       3409.266854
min          0.000000
25%       2818.106636
50%       5858.432261
75%       8751.687334
max      12016.472548
Name: energy_mismatch, dtype: float64


Scaling: This is where I turn all of the residuals into one comparable scale using standard deviations

In [120]:
scaler = StandardScaler() #creating an empty scaler object that I will train on the baseline data

baseline_residuals = baseline_dataframe[["Th_o_residual", "Tc_o_residual"]] #pulls the residual columns from the baseline data, puts them in a 2 columned table
scaler.fit(baseline_residuals) #looks at each column and calculates the mean and standard deviation of each column and stores the values in scaler
# This basically trains the model to know what baseline operating conditions (baseline data) looks like

#Check values
print("Mean:", scaler.mean_)
print("Std:", scaler.scale_)
#They almost match the baseline residual values exactly so its correct

#Now I convert the learned values into their scaled standard deviation versions
baseline_scaled = scaler.transform(baseline_residuals) #New function: transform() converts raw value into how many standard deviations away it is from the mean
print(baseline_scaled[:5]) #check

Mean: [0.01016675 0.00510927]
Std: [0.19962861 0.20979521]
[[-0.19916256  0.3240963 ]
 [ 1.13003458 -0.60483259]
 [-0.79152084 -0.9578946 ]
 [ 0.35601317 -0.79631227]
 [ 1.33623006 -0.12327855]]


Now I apply the scaler to each of the 4 fault types

In [121]:
#Fouling
fouling_residuals = fouling_dataframe[["Th_o_fouling", "Tc_o_fouling"]]
fouling_residuals.columns = ["Th_o_residual", "Tc_o_residual"]   #.columns renames to match the baseline's column names
fouling_scaled = scaler.transform(fouling_residuals)
print(fouling_scaled[:5])
print(fouling_scaled[-5:])
print()

#Drift
drift_residuals = drift_dataframe[["Th_o_drift", "Tc_o_drift"]]
drift_residuals.columns = ["Th_o_residual", "Tc_o_residual"]
drift_scaled = scaler.transform(drift_residuals)
print(drift_scaled[:5])
print(drift_scaled[-5:])
print()

#Blockage
blockage_residuals = blockage_dataframe[["Th_o_blockage", "Tc_o_blockage"]]
blockage_residuals.columns = ["Th_o_residual", "Tc_o_residual"]
blockage_scaled = scaler.transform(blockage_residuals)
print(blockage_scaled[:5])
print(blockage_scaled[-5:])
print()

#Leak
#must create a new seperate scaler as the leakage is based off of an energy balance mismatch and the Th_o and Tc_o values will look like baseline values
leak_scaler = StandardScaler()

#Also a new baseline values for the energy balance mismatch need to be created so that I dont train it on an array of zeros, this is because sensor noise at each
#side of the outlet will not be exactly the same, and if its all zeros anything thats not zero will be flagged as a fault]

sensor_noise_mdot_h_baseline = np.random.normal(loc=0, scale=0.01, size=240) #Creating the baseline sensor noise at the hot side
mdot_h_outlet_healthy = mdot_h_values + sensor_noise_mdot_h_baseline #Calculates healthy data values for mdot without (1-leak_fraction) as baseline has no leak

#Heat capacities at inlet and outlet
C_h_inlet_baseline = mdot_h_values * cp
C_h_outlet_baseline = mdot_h_outlet_healthy * cp

#Calculating Q values for inlet and outlet 
Q_inlet_baseline = C_h_inlet_baseline * (Th_i_values - baseline_dataframe["Th_o_true"])
Q_outlet_baseline = C_h_outlet_baseline * (Th_i_values - baseline_dataframe["Th_o_true"])

#Creating the baseline energy balance mismatch
baseline_dataframe["energy_mismatch"] = Q_inlet_baseline - Q_outlet_baseline

#Now fitting the leak scaler
leak_baseline_residuals = baseline_dataframe[["energy_mismatch"]]
leak_scaler.fit(leak_baseline_residuals)

print("Mean:", leak_scaler.mean_)
print("Std:", leak_scaler.scale_)

#Transforming the baseline leak data and the energy mismatch data
baseline_energy_scaled = leak_scaler.transform(leak_baseline_residuals)
print(baseline_energy_scaled[:5])

leak_energy_residuals = leak_dataframe[["

[[-0.29961512 -2.51893692]
 [-1.0962811  -2.16530594]
 [ 3.74540521  1.88156028]
 [ 1.9193569   2.91295437]
 [-5.43852987 -4.47933536]]
[[ 1.91221264 10.0139158 ]
 [-5.58785964  8.45916512]
 [-7.44664401  5.584133  ]
 [-4.76068095  5.19529029]
 [ 5.60317735 13.23420968]]

[[ 1.6446228  -1.05646648]
 [ 0.94217771  0.08340536]
 [ 0.67928709 -0.0550058 ]
 [ 0.19693992  0.33305447]
 [-0.07306388  1.66682604]]
[[-1.45187465e+01  3.79002943e-01]
 [-1.60311359e+01 -9.91080083e-03]
 [-1.72959883e+01 -6.56390596e-01]
 [-1.51340646e+01 -9.99356104e-01]
 [-1.61435125e+01 -1.18639236e+00]]

[[ 0.38717639 -2.08743245]
 [-1.49441783 -2.45444748]
 [ 3.56599753  2.82127755]
 [ 1.48224826  4.87947685]
 [-5.64951453 -6.9900639 ]]
[[68.98962964 52.69163438]
 [62.76440709 49.00628318]
 [60.55490301 46.95850445]
 [62.15566565 46.55452188]
 [71.33978042 54.94129971]]
Mean: [-47.32178419]
Std: [1101.48455928]
